# Train an Image Classifier for IGNODE

**Maintained by:** IGNODE  
**Last verified:** 2026-05-30 against PyTorch 2.4, torchvision 0.19  
**Runtime:** ~3-8 minutes on Colab's free T4 GPU; ~15-30 minutes on CPU

Train an image-classification model — predict which class an image belongs to (e.g. defect / no-defect, fruit type, animal). Uses pretrained MobileNetV2 (default) or ResNet50 backbones. Output is byte-compatible with IGNODE's in-platform image trainer — same ONNX shape, same sidecar contract.

## Quick start

### To try it right now (no setup needed)

1. **Runtime → Change runtime type → GPU** (T4 is fine; free tier)
2. **Runtime → Run all** at the top
3. Wait ~3-8 minutes for training on the sample fruits dataset (3 classes × 30 images)
4. The last cell automatically downloads `model.onnx` + sidecar JSONs to your laptop

### To train on YOUR data

Two things change:

| Step | Cell | What to do |
|---|---|---|
| 1 | **Load dataset** cell | `SAMPLE_DATASET = 'sample-fruits'` → `SAMPLE_DATASET = None` |
| 2 | Same cell | Drop your own `.tar` when the upload prompt appears — see the cell for the expected folder structure |

Your `.tar` must be a folder of class subfolders:

```
your-dataset.tar
├── ClassA/
│   ├── img_001.jpg
│   └── ...
├── ClassB/
│   └── ...
└── ClassC/
    └── ...
```

JPG, PNG, BMP, and WebP are all accepted. The class names come from the folder names. No CSV or labels.json needed — the directory structure IS the labels.

### What you get at the end

- `model.onnx` — your trained image classifier
- `class_labels.json` — the class names (in the order the model emits them)
- `preprocess_config.json` — input size, channel order, ImageNet mean/std (IGNODE's inference runtime needs this to match training-time preprocessing)

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

## 1. Install pinned dependencies

PyTorch + torchvision are pre-installed in Colab — we just pin to known-good versions and add the ONNX exporter.

In [ ]:
!pip install --quiet \
    onnx==1.21.0 \
    onnxruntime==1.23.2 \
    pillow

import sys
import torch
import torchvision
import onnx
print(f'Python:      {sys.version.split()[0]}')
print(f'PyTorch:     {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'onnx:        {onnx.__version__}')

## 2. Check the runtime

Training image classifiers is **much** faster on a GPU. If the cell below reports CPU, go to **Runtime → Change runtime type → GPU** (T4 is fine; available for free) and **Restart runtime**, then re-run from the top.

On CPU the sample dataset trains in ~15-30 min; on T4 it's ~3-8 min.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'   Total memory:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    DEVICE = torch.device('cpu')
    print('⚠ CPU only — training will be ~5x slower.')
    print('  To get a free GPU: Runtime → Change runtime type → GPU → Save → Runtime → Restart runtime')
    print('  Then re-run from the top.')

## 3. Load dataset

Ships set to the small **sample-fruits** sample (3 classes × 30 images, ~600 KB) so the notebook runs end-to-end out of the box. To use your own data, set `SAMPLE_DATASET = None` below and the cell will prompt you to upload a `.tar`.

Expected structure for your own `.tar`:

```
your-dataset.tar
├── ClassA/
│   ├── img_001.jpg  (or .png, .bmp, .webp — any common image format)
│   └── ...
└── ClassB/
    └── ...
```

On IGNODE's portal you can also create this `.tar` directly by dragging a folder into the **Create ML Job** wizard — the browser tars it client-side. The Colab notebook expects the same shape.

In [ ]:
import os
import shutil
import tarfile

# ───────── EDIT THIS ─────────
SAMPLE_DATASET = 'sample-fruits'   # set to None to upload your own .tar
# ────────────────────────────

DATASET_DIR = '/content/dataset'   # where the images will live after extraction

# Clean any prior state — re-running this cell should give a fresh dataset folder.
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

if SAMPLE_DATASET == 'sample-fruits':
    # Pull the sample-fruits folder from the public ignode-collab repo via
    # a shallow git clone (faster + smaller than wget'ing the repo tarball).
    print('Cloning sample-fruits from ignode-collab...')
    !git clone --quiet --depth 1 https://github.com/IGNODE-CONNECT/ignode-collab.git /tmp/ignode-collab
    src = '/tmp/ignode-collab/examples/images/sample-fruits'
    for cls in sorted(os.listdir(src)):
        cls_path = os.path.join(src, cls)
        if os.path.isdir(cls_path):
            shutil.copytree(cls_path, os.path.join(DATASET_DIR, cls))
    shutil.rmtree('/tmp/ignode-collab')
elif SAMPLE_DATASET is None:
    # Customer uploads a .tar with class subfolders inside.
    from google.colab import files
    print('Drop your .tar file (containing one folder per class):')
    uploaded = files.upload()
    tar_path = next(iter(uploaded.keys()))
    with tarfile.open(tar_path) as tf:
        tf.extractall('/content/_raw')
    # The customer's tar might wrap everything in a top-level folder. Find the
    # first directory that has class subfolders inside it.
    root = '/content/_raw'
    candidates = [root] + [os.path.join(root, d) for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    src = None
    for c in candidates:
        subdirs = [d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))]
        if len(subdirs) >= 2:
            src = c
            break
    if src is None:
        raise ValueError('Could not find class subfolders in the uploaded .tar. Make sure your tar contains one folder per class.')
    for cls in sorted(os.listdir(src)):
        cls_path = os.path.join(src, cls)
        if os.path.isdir(cls_path):
            shutil.copytree(cls_path, os.path.join(DATASET_DIR, cls))
    shutil.rmtree(root)
    os.remove(tar_path)
else:
    raise ValueError(f'Unknown SAMPLE_DATASET={SAMPLE_DATASET!r}. Use "sample-fruits" or None.')

# Surface what landed.
classes = sorted(os.listdir(DATASET_DIR))
print(f'\nDataset at {DATASET_DIR}:')
for cls in classes:
    n = len([f for f in os.listdir(os.path.join(DATASET_DIR, cls)) if not f.startswith('.')])
    print(f'  {cls}: {n} images')

## 4. Inspect the dataset

Renders 4 thumbnails per class so you can sanity-check that the right images landed under the right labels. If any thumbnail looks wrong (e.g. an image of a banana in the Pineapple folder), fix the source data before training.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

PER_CLASS = 4
fig, axes = plt.subplots(len(classes), PER_CLASS, figsize=(PER_CLASS * 2, len(classes) * 2))
if len(classes) == 1:
    axes = [axes]
for row, cls in enumerate(classes):
    files_list = sorted(os.listdir(os.path.join(DATASET_DIR, cls)))[:PER_CLASS]
    for col in range(PER_CLASS):
        ax = axes[row][col] if len(classes) > 1 else axes[col]
        ax.axis('off')
        if col < len(files_list):
            img = Image.open(os.path.join(DATASET_DIR, cls, files_list[col]))
            ax.imshow(img)
            if col == 0:
                ax.set_ylabel(cls, fontsize=11, rotation=0, labelpad=40, ha='right', va='center')
plt.tight_layout()
plt.show()

## 5. Settings

**Edit this cell if you want to change the backbone, training duration, or batch size.** Defaults work well for the sample dataset.

### Backbones

| `BACKBONE` | Size | Speed | Accuracy |
|---|---|---|---|
| `'mobilenet_v2'` | ~13 MB | Fast | Good for most cases (default) |
| `'resnet50'` | ~98 MB | Medium | Stronger; use when you have ≥100 images per class |

### Other settings

- `EPOCHS` — passes through the training set. More = better accuracy, longer training. 10-15 is a good starting point.
- `BATCH_SIZE` — images per training step. 16 is fine for free Colab; reduce to 8 if you hit memory errors.
- `LEARNING_RATE` — step size. 0.001 is a safe default for transfer learning.

In [ ]:
# ───────── EDIT THESE ─────────
BACKBONE = 'mobilenet_v2'    # 'mobilenet_v2' or 'resnet50'
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 0.001

TEST_SIZE = 0.2              # fraction held out for evaluation
RANDOM_SEED = 42             # locks the result
INPUT_SIZE = 224             # square; both backbones expect 224x224
# ──────────────────────────────

if BACKBONE not in ('mobilenet_v2', 'resnet50'):
    raise ValueError(f"BACKBONE must be 'mobilenet_v2' or 'resnet50', got {BACKBONE!r}")

import torch, random, numpy as np
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print(f'Backbone:        {BACKBONE}')
print(f'Epochs:          {EPOCHS}')
print(f'Batch size:      {BATCH_SIZE}')
print(f'Learning rate:   {LEARNING_RATE}')
print(f'Input size:      {INPUT_SIZE}x{INPUT_SIZE}')

## 6. Prepare data loaders

Build PyTorch DataLoaders for training + validation. Images are resized to 224×224, converted to RGB CHW tensors, and normalized with the ImageNet mean/std (the standard for pretrained backbones).

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# Same preprocessing pipeline IGNODE's in-platform image trainer uses, so the
# trained model produces equivalent predictions on equivalent inputs at
# inference time.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset = datasets.ImageFolder(DATASET_DIR, transform=eval_transform)
class_labels = full_dataset.classes
print(f'Classes ({len(class_labels)}): {class_labels}')
print(f'Total images: {len(full_dataset)}')

# Split. Use a generator for the seeded split so results are reproducible.
n_test = max(1, int(len(full_dataset) * TEST_SIZE))
n_train = len(full_dataset) - n_test
gen = torch.Generator().manual_seed(RANDOM_SEED)
train_set, test_set = random_split(full_dataset, [n_train, n_test], generator=gen)

# Apply augmentation to the training split only.
train_set.dataset.transform = train_transform
test_set.dataset.transform = eval_transform

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {n_train} images. Test: {n_test} images.')

## 7. Train

Load the pretrained backbone, replace the final classification layer with one sized for our classes, and fine-tune. Progress per epoch shows training loss + validation accuracy.

In [ ]:
import time
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Build the model — pretrained backbone with the final classifier swapped
# to match our class count.
if BACKBONE == 'mobilenet_v2':
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_labels))
elif BACKBONE == 'resnet50':
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, len(class_labels))

model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'val_accuracy': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    total_loss, n_seen = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        n_seen += images.size(0)
    train_loss = total_loss / n_seen

    # Validation accuracy at the end of each epoch.
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_accuracy = correct / total

    history['train_loss'].append(train_loss)
    history['val_accuracy'].append(val_accuracy)
    print(f'Epoch {epoch:2d}/{EPOCHS}  '
          f'train_loss={train_loss:.4f}  val_accuracy={val_accuracy:.3f}  '
          f'({time.time() - t0:.1f}s)')

# Plot the training curves.
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history['train_loss'], marker='o')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training loss'); ax1.grid(alpha=0.3)
ax2.plot(history['val_accuracy'], marker='o', color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation accuracy'); ax2.set_ylim(0, 1); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Evaluate

Confusion matrix on the held-out test set. The diagonal is correct predictions. Off-diagonal bright spots show which class pairs the model confuses.

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(dim=1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.numpy().tolist())

print(f'Test-set accuracy: {accuracy_score(y_true, y_pred):.3f}\n')
print(classification_report(y_true, y_pred, target_names=class_labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=class_labels).plot(
    ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False,
)
ax.set_title('Confusion Matrix (held-out test set)')
plt.tight_layout(); plt.show()

## 9. Export to ONNX

Convert the trained PyTorch model to ONNX with the exact contract IGNODE's inference runtime expects:

- Opset 18 (matches IGNODE's pinned ONNX stack)
- Output name `Score` (so IGNODE's classifier loader finds the prediction tensor)
- Dynamic batch dimension so the runtime can batch predict requests

In [ ]:
import torch

model.eval()
dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    'model.onnx',
    input_names=['input'],
    output_names=['Score'],
    opset_version=18,
    do_constant_folding=True,
    dynamic_axes={
        'input': {0: 'batch_size'},
        'Score': {0: 'batch_size'},
    },
)

import os
print(f'Saved model.onnx ({os.path.getsize("model.onnx") / (1024 * 1024):.1f} MB)')

## 10. Write sidecar files

IGNODE's inference runtime reads two sidecars alongside the model:

- `class_labels.json` — class names in the order the model emits them
- `preprocess_config.json` — input size, channel order, image format (CHW), normalization. Tells the inference pod to preprocess images at predict time exactly the way we did at training time.

In [ ]:
import json

with open('class_labels.json', 'w') as f:
    json.dump(class_labels, f, indent=2)

preprocess = {
    'input_size': [INPUT_SIZE, INPUT_SIZE],
    'mean': IMAGENET_MEAN,
    'std':  IMAGENET_STD,
    'channel_order': 'RGB',
    'image_format':  'CHW',
    'rescale':       'imagenet',
    'resize_method': 'center_crop',
    'resize_short_side': 256,
    '_backbone': BACKBONE,   # audit only — IGNODE's loader ignores this
}
with open('preprocess_config.json', 'w') as f:
    json.dump(preprocess, f, indent=2)

print('class_labels.json:')
print(json.dumps(class_labels, indent=2))
print('\npreprocess_config.json:')
print(json.dumps(preprocess, indent=2))

## 11. Download

In [ ]:
from google.colab import files
files.download('model.onnx')
files.download('class_labels.json')
files.download('preprocess_config.json')

## 12. Upload to IGNODE

1. **Integrations → ML Factory** in your IGNODE portal
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model** (or the **Add Model** dropdown → **Upload Custom Model**)
4. Drop your `model.onnx` and fill in the metadata form:
    - **Task Type:** `Image Classification`
    - **Class Labels:** paste from `class_labels.json`
    - **Image Preprocessing:** values from `preprocess_config.json` (the wizard auto-detects most of these, but verify they match)
5. Click **Upload**, then **Open in Playground** to test against a held-out image.

Image classification models can only deploy to **ML Inference (UINF)** — Workspaces don't support image inference. The wizard will hide the Workspace target option.

---

## Reusing this notebook for your own data

Two changes switch to your own dataset:

```python
# In the "Load dataset" cell:
SAMPLE_DATASET = None   # was 'sample-fruits'
# Then drop your own .tar when the upload prompt appears
```

Your `.tar` must contain one folder per class, with images inside each folder. JPG / PNG / BMP / WebP all accepted.

### Optional tweaks (Settings cell)

| Want to change | Edit |
|---|---|
| Stronger backbone (more accurate, slower) | `BACKBONE = 'resnet50'` |
| More training | `EPOCHS = 20` |
| Smaller batches (if OOM) | `BATCH_SIZE = 8` |
| Slower learning (more stable) | `LEARNING_RATE = 0.0005` |
| Larger held-out test split | `TEST_SIZE = 0.3` |

### Tips

- **Need at least ~30 images per class** to get reasonable accuracy. 100+ is much better.
- **Class imbalance** (e.g. 200 of one class, 10 of another) will bias the model. Balance the dataset before training, or oversample the minority class.
- **Class folder names become labels** — pick human-readable names (`defective` not `class_0`).
- **The sample dataset is intentionally tiny** (90 images) to demonstrate the workflow. Don't expect strong accuracy from it; expect ~60-80% on the held-out split.

### Common errors

| Error | Fix |
|---|---|
| CUDA out of memory | Reduce `BATCH_SIZE` to 8 (or 4 on very limited GPUs) |
| `Could not find class subfolders in the uploaded .tar` | Your tar wraps the classes in an extra folder — that's fine, the notebook unwraps it. If you see this error, double-check the tar structure with `tar tf your.tar` |
| Validation accuracy stuck at 33% (3 classes) / 50% (2 classes) | Model isn't learning — try a different backbone, more epochs, or check class balance |
| Upload rejected: "image_deploy_workspace_unsupported" | You picked a Workspace target — image models deploy to ML Inference (UINF) only. Re-select the target. |

### Bringing this code into your own project

Almost all the code is vanilla PyTorch. The only Colab-specific pieces are `google.colab.files.upload()` and `google.colab.files.download()`. To run outside Colab:

1. Replace the upload block in the load-dataset cell with `tarfile.open('/your/local/dataset.tar').extractall(DATASET_DIR)`
2. Replace the `files.download(...)` calls in the download cell with normal file moves / S3 puts / whatever your project does
3. Everything else is portable